# Bible Study Virtual Assistant — Class Project Demo

An LLM-based virtual assistant that answers Bible-study questions using **structured tools**, not just model memory.

**Models compared**
- Smaller: `microsoft/Phi-3.5-mini-instruct`
- Larger: `mistralai/Mistral-7B-Instruct-v0.3` (4-bit on Colab)

**Tools**
- Bible verse lookup + TF-IDF search (KJV, public domain)
- BEMA Discipleship podcast transcript retrieval (TF-IDF)
- DuckDuckGo web search for historical context

**Prompting techniques**: zero-shot, few-shot, chain-of-thought

**Security**: 5 prompt-injection / abuse tests

---

> ### ⚠️ Where to run this
>
> - **Sections 1–3 + 3a (MockLLM)** run on any laptop, no GPU needed.
> - **Sections 3b, 4–8 require a GPU.** Phi-3.5 fits in any T4; Mistral-7B needs 4-bit quantization to fit a T4.
>   - In Colab: **Runtime → Change runtime type → T4 GPU** before running section 4.
>   - Models are loaded **sequentially** (Phi-3.5 first → freed → Mistral) to stay under the free T4's ~15 GB VRAM ceiling.
>   - On a Mac with < 16 GB RAM the real LLMs will not load; use Colab.

## 1. Setup (Colab)

Skip this cell if running locally with the repo already cloned.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/emilymoberly/bible-study-va.git"
REPO_DIR = "bible-study-va"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # local: assume notebook is run from notebooks/ inside the repo
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print("cwd:", os.getcwd())

## 2. Build the data

The pipeline collects 6 source types into one labeled corpus:

| Step | Time | Output |
|---|---|---|
| `load_bible.py` | ~10s | KJV verses (31 K) |
| `scrape_bema.py` | ~9 min | 259 podcast transcripts |
| `scrape_bema_pages.py` | ~12 min | 507 episode summaries + study tools + 5 site pages |
| `scrape_youtube.py` | ~3 min | captions for videos linked from BEMA pages (only those with public captions; YouTube IP-rate-limits aggressively, so coverage is partial) |
| `build_corpus.py` | ~10s | unified `data/corpus/corpus.jsonl` with labeled, deduped, verse-tagged chunks |

- Download the KJV Bible JSON (~6 MB, one-time, ~5 sec).
- Scrape BEMA transcripts via the official Fireside JSON feed. The full
  scrape is ~9 min and produces ~259 transcripts (~1.65 M words). For a
  faster first run pass `--max-episodes 30`.

In [ ]:
!python scripts/load_bible.py
!python scripts/scrape_bema.py            # full corpus; or --max-episodes 30 for a 1-min smoke run
!python scripts/scrape_bema_pages.py      # episode summaries + study tools + 5 static site pages
!python scripts/scrape_youtube.py         # YouTube captions; will skip videos without public captions
!python scripts/build_corpus.py           # unify everything into data/corpus/corpus.jsonl

## 3. Smoke-test the tools (no LLM yet)

In [ ]:
from src.bible_tool import BibleTool
from src.bema_tool import BemaTool

bible = BibleTool("data/bible/bible.json")
bema = BemaTool("data/bema/transcripts", "data/bema/episodes.json")

print("Verse lookup — John 3:16:")
for v in bible.lookup("John 3:16"):
    print(" ", v)

print("\nVerse search — 'babylon':")
for hit in bible.search("babylon great fallen", k=3):
    print(f"  ({hit.score:.2f}) {hit.item}")

print(f"\nBEMA chunks indexed: {len(bema.chunks)}")
print("BEMA search — 'creation story':")
for hit in bema.search("creation story Genesis Eastern thinking", k=2):
    print(f"  ({hit.score:.2f}) {hit.item}")

## 3a. (Optional, no-GPU) Quick agent demo with MockLLM

If you're on a laptop without a GPU and just want to see the **agent pipeline** work end-to-end (routing → retrieval → prompt assembly), run this cell. It uses a stand-in LLM that doesn't load any weights. Skip if you're on Colab with a GPU and want the real models.

In [ ]:
from src.agent import Agent
from src.llm import MockLLM

# Build the corpus first (one-time, ~10 sec).
!python scripts/build_corpus.py

# Agent now reads from the unified labeled corpus (Bible + BEMA + YouTube).
agent = Agent("data/corpus/corpus.jsonl", enable_web=False)
print("Corpus stats:", agent.corpus.stats())

mock = MockLLM()
ans = agent.answer(
    "What is a chiasm and what are examples in Deuteronomy?",
    model=mock,
    technique="zero_shot",
)
print("\nRouting:", ans.routing.reason)
print(f"Latency: {ans.latency_s*1000:.0f} ms")
print("Top sources:")
for s in ans.sources[:3]:
    print(" -", s.label)
print("\nAnswer:\n", ans.text)

## 3b. Hugging Face login (required for Mistral)

`mistralai/Mistral-7B-Instruct-v0.3` is a **gated** model — Hugging Face requires an authenticated download. One-time setup:

1. Accept the license at [huggingface.co/mistralai/Mistral-7B-Instruct-v0.3](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) (click "Agree and access repository" at the top of the page).
2. Create a **Read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. In Colab: click the 🔑 **key icon** in the left sidebar → **+ Add new secret** → name it `HF_TOKEN`, paste your `hf_...` token, toggle **Notebook access** ON for this notebook.

Then run the cell below — you should see your username printed.

In [ ]:
from huggingface_hub import login, whoami

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except (ImportError, KeyError):
    import os
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Logged in as:", whoami()["name"])
else:
    print("No HF_TOKEN found.")
    print("Phi-3.5 will still work (public model), but Mistral download will be"
          " rate-limited or fail. Add HF_TOKEN to Colab Secrets and re-run.")

## 4. Load Phi-3.5 (sequential approach, T4-safe)

Memory budget on a free Colab T4 (~15 GB VRAM):
- **Phi-3.5-mini** in fp16/bf16 ≈ **7.6 GB**
- **Mistral-7B** in 4-bit (bitsandbytes) ≈ **5 GB**

In theory both fit, but in practice fragmentation + activations push you over the cliff on a free T4. We load **one at a time**:

1. Load Phi-3.5 (this cell) → run sample answer + comparison + security tests.
2. Free Phi-3.5, load Mistral → run Mistral comparison.
3. Concatenate the two result tables for the chart and quality scoring.

The first time you run this cell it downloads the Phi-3.5 weights from Hugging Face (~7.6 GB, ~1–2 min). Subsequent runs in the same Colab session load instantly from disk cache.

In [ ]:
import torch
from src.llm import LLM

phi3 = LLM.load("phi3")
print(f"Phi-3.5 on: {phi3.model.device}")
print(f"GPU mem used: {torch.cuda.memory_allocated()/1e9:.2f} GB"
      f" / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

# Smoke test: if this errors, the model didn't load right.
print("\nphi3 says:", phi3.generate("Say hi in 5 words.", max_new_tokens=20).text)

## 5. Build the agent and answer a sample question

In [ ]:
from src.agent import Agent

# Re-create the agent with web search enabled now that we have a real model.
agent = Agent("data/corpus/corpus.jsonl", enable_web=True)

demo = agent.answer(
    "What are hidden meanings and symbols behind Babylon in the Bible?",
    model=phi3,
    technique="zero_shot",
)

print("ROUTING:", demo.routing.reason)
print(f"LATENCY: {demo.latency_s:.1f}s | tokens in/out: {demo.tokens_in}/{demo.tokens_out}")
print("\nANSWER:\n", demo.text)
print("\nSOURCES:")
for s in demo.sources[:5]:
    print(" -", s.label)

## 6. Model + prompting-technique comparison

Run our 5 example questions through both models and all three techniques, then collect a results table.

In [ ]:
import pandas as pd

QUESTIONS = [
    "What are hidden meanings and symbols behind Babylon in the Bible?",
    "What was the common Jewish cultural acceptance of wine?",
    "How did the Roman Empire clash with the Jews?",
    "What is a chiasm and what are examples in Deuteronomy?",
    "What is the difference between a Pharisee and a teacher of the law?",
]
TECHNIQUES = ["zero_shot", "few_shot", "chain_of_thought"]


def run_comparison(model, model_name: str) -> pd.DataFrame:
    """Run all questions x techniques against one model. Returns a DataFrame."""
    rows = []
    total = len(QUESTIONS) * len(TECHNIQUES)
    n = 0
    for q in QUESTIONS:
        for t in TECHNIQUES:
            n += 1
            ans = agent.answer(q, model=model, technique=t, max_new_tokens=350)
            rows.append({
                "question": q[:60] + "...",
                "model": model_name,
                "technique": t,
                "latency_s": round(ans.latency_s, 2),
                "tokens_out": ans.tokens_out,
                "answer": ans.text[:300] + "...",
            })
            print(f"  [{n}/{total}] {model_name} {t}  ({ans.latency_s:.1f}s)")
    return pd.DataFrame(rows)


# Phi-3.5 first (it's already loaded). ~5-10 min on T4.
df_phi = run_comparison(phi3, "phi3")
df_phi

In [ ]:
import gc, torch
from src.llm import LLM

# Free Phi-3.5 to make room for Mistral on the T4.
del phi3
gc.collect()
torch.cuda.empty_cache()
print(f"GPU mem after free: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Load Mistral-7B in 4-bit (bitsandbytes). ~3-5 min download first time, ~30 sec from disk after.
mistral = LLM.load("mistral")
print(f"Mistral on: {mistral.model.device}")
print(f"GPU mem used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Smoke test before the long loop.
print("\nmistral says:", mistral.generate("Say hi in 5 words.", max_new_tokens=20).text)

# Full Mistral comparison. ~10-15 min on T4 (4-bit is slower per token than fp16).
df_mistral = run_comparison(mistral, "mistral")

# Combine so the chart + quality cells "just work".
df = pd.concat([df_phi, df_mistral], ignore_index=True)
print(f"\nTotal rows: {len(df)} ({len(df_phi)} phi3 + {len(df_mistral)} mistral)")
df

In [ ]:
import matplotlib.pyplot as plt

agg = df.groupby(["model", "technique"])["latency_s"].mean().unstack()
ax = agg.plot(kind="bar", figsize=(8, 4))
ax.set_title("Mean response time by model and prompting technique")
ax.set_ylabel("seconds")
ax.set_xlabel("model")
plt.tight_layout()
plt.show()

## 7. Manual quality scoring

After reading the answers in the table above, fill in a 1–5 quality score for each row in the cell below. This is the qualitative half of the rubric.

In [ ]:
df["quality"] = 0  # <-- edit per row, e.g. df.loc[0, 'quality'] = 4
df.groupby(["model", "technique"])["quality"].mean().unstack()

## 8. Prompt-injection / security tests

Run all 5 attacks against the currently-loaded model and see whether each is defended. By the time we get here, Mistral-7B is loaded (Phi-3.5 was freed earlier to make room). If you want a phi3-vs-mistral defense comparison, free Mistral and reload phi3 (`del mistral; gc.collect(); torch.cuda.empty_cache(); phi3 = LLM.load("phi3")`) and re-run this cell.

In [ ]:
from src.security import run_all

# Use whichever real model is currently in memory (mistral by default after the
# sequential load above; phi3 if you reloaded it per the markdown note).
_security_model = mistral if "mistral" in dir() else phi3
_security_name = "mistral" if _security_model is mistral else "phi3"
print(f"Running 5 prompt-injection attacks against: {_security_name}\n")

results = run_all(agent, _security_model, technique="zero_shot")
for r in results:
    print(r)
    print("-" * 80)

successes = sum(r.succeeded for r in results)
print(f"\nDefense summary ({_security_name}): "
      f"{len(results) - successes}/{len(results)} attacks blocked")

## 9. Discussion / limitations

Things to write up in the report:

- Where Phi-3.5 was clearly weaker / stronger than Mistral-7B
- How chain-of-thought changed answer quality vs latency
- Which attack(s) succeeded and what an additional defense would look like
  (input filtering, output filtering, separating data from instructions, etc.)
- TF-IDF vs embeddings: cases where TF-IDF missed clearly-relevant context

## 10. Launch the Gradio UI

Run this cell to get an interactive web UI. In Colab, `share=True` returns
a public `*.gradio.live` URL you can open in a new tab (good for the demo).

The UI lets you:
- type a question, theme, or verse like `John 3:16`
- filter sources (Bible only, BEMA only, YouTube only, all sources)
- pick a prompting technique (zero-shot / few-shot / chain-of-thought)
- pick a model (`mock`, `phi3`, or `mistral`) — `phi3` and `mistral` are
  lazy-loaded the first time you click "Ask" with them selected
- see the top sources used as evidence with clickable URLs

In [ ]:
from src.app import build_app

# In Colab, share=True is what gives you a public link.
build_app().launch(share=True)